# AgentCore MCP Gateway Resource Management Operations

This notebook demonstrates comprehensive **AgentCore MCP Gateway** lifecycle management using the AgentCoreGatewayClient.

## What You'll Learn

- **Create**: MCP Gateway resources with configurable protocols and authorization
- **Read**: Get gateway details and configuration
- **Update**: Modify gateway settings and configurations
- **Delete**: Clean up gateway resources
- **List**: Enumerate all gateway resources
- **Wait Operations**: Monitor gateway creation and deletion status

## AgentCore MCP Gateway Operations

You can find boto3 (Python) AgentCore control plane gateway operations on this page:

https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agentcore-control.html

* AgentCore MCP Gateways provide a managed endpoint for Model Context Protocol (MCP) servers, enabling secure communication between agents and external tools/resources.

* The classes provided here under `src/agentic_platform/service/agentcore/mcp_gateway` wrap the AgentCore SDK service calls for gateway management.

* The goal of the AgentCore support in this project is to demonstrate how to wrap those AgentCore calls in resource management services and deploy those resource management services to AWS via Terraform.

* The gateway operations include:
    * create_gateway: Create a new MCP gateway with specified protocol and authorization configuration
    * get_gateway: Get information about a gateway by gatewayId
    * delete_gateway: Delete the gateway resource
    * list_gateways: List all gateway resources
    * update_gateway: Update gateway configuration

The example AgentCoreGatewayClient in this project currently supports these operations:

- `CREATE` - Create new MCP gateways
- `GET` - Retrieve gateway details
- `UPDATE` - Update gateway configurations
- `DELETE` - Delete gateway resources
- `LIST` - List all gateways
- `WAIT_FOR_CREATE` - Wait for gateway creation
- `WAIT_FOR_DELETE` - Wait for gateway deletion

## Key Concepts

**AgentCore MCP Gateway**: A managed endpoint that provides secure access to Model Context Protocol (MCP) servers. Gateways handle authentication, authorization, and protocol translation.

**Protocol Type**: Currently supports MCP (Model Context Protocol) for connecting to MCP servers.

**Authorization**: Configurable authorization mechanisms including CUSTOM_JWT for secure access control.

**Gateway URL**: The endpoint URL where the gateway can be accessed by clients.

## Prerequisites

- AWS credentials with AgentCore permissions
- IAM role ARN with appropriate permissions for the gateway

### First let's look at the AgentCore Gateway Client from this project.

In [1]:
# %load ../../src/agentic_platform/service/agentcore/mcp_gateway/client/agentcore_gateway_client.py
"""
AWS Lambda function to provision and manage Bedrock AgentCore Gateway resources.

This function is used by Terraform to create and update Bedrock AgentCore gateway
capabilities for the Agentic Platform.

Usage:
  - The function is invoked by Terraform with appropriate configuration parameters
  - It creates or updates gateway resources using the boto3 SDK
  - It handles provision action for now

Environment Variables:
  - REGION: AWS region for Bedrock AgentCore resources
"""

import boto3
import json
import logging
import os
import time
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

# Configure logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Get environment variables
REGION = os.environ.get('REGION', 'us-west-2')

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
ssm_client = boto3.client('ssm', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)

class AgentCoreGatewayClient:

    @staticmethod
    def _create_default_iam_role(gateway_name: str) -> str:
        """
        Create a default IAM role for the AgentCore Gateway.
        
        Args:
            gateway_name: Name of the gateway to create role for
            
        Returns:
            ARN of the created IAM role
        """
        # Generate unique role name
        role_name = f"AgentCoreGateway-{gateway_name}-{uuid.uuid4().hex[:8]}"
        
        # Trust policy allowing bedrock-agentcore and lambda services to assume the role
        trust_policy = {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Principal": {
                        "Service": [
                            "lambda.amazonaws.com",
                            "bedrock-agentcore.amazonaws.com"
                        ]
                    },
                    "Action": "sts:AssumeRole"
                }
            ]
        }
        
        # Role policy allowing lambda invocation
        role_policy = {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Action": "lambda:InvokeFunction",
                    "Resource": "*",
                    "Effect": "Allow"
                }
            ]
        }
        
        try:
            print(f"Creating IAM role: {role_name}")
            
            # Create the IAM role
            create_role_response = iam_client.create_role(
                RoleName=role_name,
                AssumeRolePolicyDocument=json.dumps(trust_policy),
                Description=f"Default IAM role for AgentCore Gateway {gateway_name}",
                Tags=[
                    {
                        'Key': 'Purpose',
                        'Value': 'AgentCoreGateway'
                    },
                    {
                        'Key': 'GatewayName',
                        'Value': gateway_name
                    }
                ]
            )
            
            role_arn = create_role_response['Role']['Arn']
            print(f"Created IAM role with ARN: {role_arn}")
            
            # Attach the inline policy to the role
            policy_name = f"AgentCoreGatewayPolicy-{gateway_name}"
            iam_client.put_role_policy(
                RoleName=role_name,
                PolicyName=policy_name,
                PolicyDocument=json.dumps(role_policy)
            )
            
            print(f"Attached policy {policy_name} to role {role_name}")
            
            # Wait a moment for IAM consistency
            print("Waiting for IAM role to be available...")
            time.sleep(10)
            
            return role_arn
            
        except Exception as e:
            logger.error(f"Error creating default IAM role: {str(e)}")
            raise e

    @staticmethod
    def create_gateway(
        name: str,
        role_arn: Optional[str] = None,
        protocol_type: str = 'MCP',
        description: Optional[str] = None,
        protocol_configuration: Optional[Dict[str, Any]] = None,
        authorizer_type: str = 'CUSTOM_JWT',
        authorizer_configuration: Optional[Dict[str, Any]] = None,
        kms_key_arn: Optional[str] = None,
        exception_level: Optional[str] = None,
        client_token: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Create a Bedrock AgentCore Gateway resource.
        
        Args:
            name: The name of the gateway. Must be unique within your account.
            role_arn: Optional Amazon Resource Name (ARN) of the IAM role that provides 
                     permissions for the gateway to access AWS services. If not provided,
                     a default role will be created.
            protocol_type: The protocol type for the gateway. Currently supports MCP.
            description: Optional description of the gateway.
            protocol_configuration: Configuration settings for the protocol.
            authorizer_type: The type of authorizer to use for the gateway.
            authorizer_configuration: The authorizer configuration for the Gateway.
            kms_key_arn: Optional ARN of the KMS key used to encrypt data.
            exception_level: The verbosity of exception messages (DEBUG or None).
            client_token: Optional unique token to ensure idempotency.
            
        Returns:
            Dictionary with the gateway creation result
        """
        
        print(f"Creating AgentCore Gateway {name} with protocol {protocol_type}")
        try:
            # Create default IAM role if role_arn is not provided
            if role_arn is None:
                print("No role_arn provided, creating default IAM role...")
                role_arn = AgentCoreGatewayClient._create_default_iam_role(name)
                print(f"Created default IAM role: {role_arn}")
            
            # Prepare create parameters
            create_params = {
                'name': name,
                'roleArn': role_arn,
                'protocolType': protocol_type,
                'authorizerType': authorizer_type
            }
            
            # Add optional parameters if provided
            if description:
                create_params['description'] = description
            if client_token:
                create_params['clientToken'] = client_token
            if protocol_configuration:
                create_params['protocolConfiguration'] = protocol_configuration
            if authorizer_configuration:
                create_params['authorizerConfiguration'] = authorizer_configuration
            else:
                create_params['authorizerConfiguration'] = {
                    'customJWTAuthorizer': {
                        'discoveryUrl': os.getenv('COGNITO_DISCOVERY_URL'),
                        'allowedClients': [
                            os.getenv('COGNITO_USER_POOL_CLIENT_ID')
                        ]
                    }
                }
            if kms_key_arn:
                create_params['kmsKeyArn'] = kms_key_arn
            if exception_level:
                create_params['exceptionLevel'] = exception_level
            
            print(f"Creating gateway with parameters: {create_params}")
            
            # Create gateway using the control plane client
            create_response = agentcore_control_client.create_gateway(**create_params)
            
            print(f"Got create response: {create_response}")
            gateway_id = create_response['gatewayId']
            
            print(f"Creating AgentCore gateway resource with ID: {gateway_id}")
            print("Waiting for gateway creation to complete.")
            
            result = AgentCoreGatewayClient.wait_for_gateway_creation(gateway_id)
            print(f"Gateway creation result: {result}")
            
            status = agentcore_control_client.get_gateway(
                gatewayIdentifier=gateway_id
            )['status']

            print(f"Gateway {gateway_id} status {status}")
            if status not in ['READY', 'CREATING']:
                raise Exception(f'Failed to create gateway {name}')
            
            logger.info("AgentCore Gateway provisioning completed successfully")
            
            return {
                'gateway_id': gateway_id,
                'gateway_arn': create_response['gatewayArn'],
                'gateway_url': create_response['gatewayUrl'],
                'name': create_response['name'],
                'status': create_response['status'],
                'created_at': create_response['createdAt'].isoformat() if 'createdAt' in create_response else None,
                'updated_at': create_response['updatedAt'].isoformat() if 'updatedAt' in create_response else None
            }
        
        except Exception as create_error:
            print(f"ERROR: {str(create_error)}")
            if "already exists" in str(create_error):
                print(f"Gateway {name} already exists")
                gateways = agentcore_control_client.list_gateways()['items']
                for gateway in gateways:
                    if gateway['name'] == name:
                        print(f"Found existing gateway {gateway}")
                        # Get full gateway details using get_gateway
                        full_gateway = agentcore_control_client.get_gateway(
                            gatewayIdentifier=gateway['gatewayId']
                        )
                        return {
                            'gateway_id': full_gateway['gatewayId'],
                            'gateway_arn': full_gateway['gatewayArn'],
                            'gateway_url': full_gateway.get('gatewayUrl', ''),
                            'name': full_gateway['name'],
                            'status': full_gateway['status'],
                            'created_at': full_gateway['createdAt'].isoformat() if 'createdAt' in full_gateway else None,
                            'updated_at': full_gateway['updatedAt'].isoformat() if 'updatedAt' in full_gateway else None
                        }
            else:
                logger.error(f"Could not create gateway resource: {str(create_error)}")
                raise create_error
            
    @staticmethod
    def delete_gateway(gateway_id: str) -> Dict[str, Any]:
        """
        Delete a Bedrock AgentCore Gateway resource.
        
        Args:
            gateway_id: ID of the gateway resource to delete
            
        Returns:
            Dictionary indicating success
        """
        print(f"Deleting gateway with id {gateway_id}")
        try:
            # Delete the gateway resource 
            agentcore_control_client.delete_gateway(
                gatewayIdentifier=gateway_id
            )
            
            print(f"Successfully deleted gateway resource with ID: {gateway_id}")
            return {'gateway_id': gateway_id, 'status': 'DELETING'}
            
        except Exception as e:
            logger.error(f"Error deleting AgentCore Gateway resource: {str(e)}")
            raise e

    @staticmethod
    def get_gateway(gateway_id: str) -> Dict[str, Any]:
        """
        Get a Bedrock AgentCore Gateway resource.
        
        Args:
            gateway_id: ID of the gateway resource to retrieve
            
        Returns:
            Dictionary with gateway details
        """
        print(f"Retrieving gateway with id {gateway_id}")
        try:
            # Get the gateway resource 
            response = agentcore_control_client.get_gateway(
                gatewayIdentifier=gateway_id
            )
            
            print(f"Successfully retrieved gateway resource with ID: {gateway_id}")
            return {
                'gateway_id': response['gatewayId'],
                'gateway_arn': response['gatewayArn'],
                'gateway_url': response.get('gatewayUrl', ''),
                'name': response['name'],
                'description': response.get('description', ''),
                'status': response['status'],
                'role_arn': response['roleArn'],
                'protocol_type': response['protocolType'],
                'protocol_configuration': response.get('protocolConfiguration', {}),
                'authorizer_type': response['authorizerType'],
                'authorizer_configuration': response.get('authorizerConfiguration', {}),
                'created_at': response['createdAt'].isoformat() if 'createdAt' in response else None,
                'updated_at': response['updatedAt'].isoformat() if 'updatedAt' in response else None
            }
            
        except Exception as e:
            logger.error(f"Error retrieving AgentCore Gateway resource: {str(e)}")
            raise e
    
    @staticmethod
    def list_gateways(
        max_results: Optional[int] = None,
        next_token: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        List AgentCore Gateway resources.
        
        Args:
            max_results: Maximum number of results to return
            next_token: Token for pagination
            
        Returns:
            Dictionary with a list of Gateway resources
        """
        print(f"Listing gateways with max_results={max_results}, next_token={next_token}")
        try:
            # Prepare parameters for list_gateways call
            list_params = {}
            if max_results:
                list_params['maxResults'] = max_results
            if next_token:
                list_params['nextToken'] = next_token
                
            print(f"Listing gateways with params: {list_params}")
            
            # List all gateway resources
            response = agentcore_control_client.list_gateways(**list_params)
            gateways = response.get('items', [])
            
            print(f"Successfully retrieved {len(gateways)} gateway resources")
            
            gateway_entries = []
            for gateway in gateways:
                print(f"Got gateway: {gateway}")
                entry = {
                    'gateway_id': gateway['gatewayId'],
                    'gateway_arn': gateway.get('gatewayArn', ''),
                    'gateway_url': gateway.get('gatewayUrl', ''),
                    'name': gateway['name'],
                    'status': gateway['status'],
                    'created_at': gateway['createdAt'].isoformat() if 'createdAt' in gateway else None,
                    'updated_at': gateway['updatedAt'].isoformat() if 'updatedAt' in gateway else None
                }
                gateway_entries.append(entry)
                
            print(f"list_gateways returning gateways {gateway_entries}")
            return {
                'gateways': gateway_entries,
                'next_token': response.get('nextToken')
            }
            
        except Exception as e:
            logger.error(f"Error listing AgentCore Gateway resources: {str(e)}")
            raise e

    @staticmethod
    def update_gateway(
        gateway_id: str,
        name: Optional[str] = None,
        description: Optional[str] = None,
        role_arn: Optional[str] = None,
        protocol_configuration: Optional[Dict[str, Any]] = None,
        authorizer_configuration: Optional[Dict[str, Any]] = None,
        kms_key_arn: Optional[str] = None,
        exception_level: Optional[str] = None,
        client_token: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Update a Bedrock AgentCore Gateway resource.
        
        Args:
            gateway_id: ID of the gateway resource to update
            name: Optional new name for the gateway
            description: Optional new description for the gateway
            role_arn: Optional new IAM role ARN
            protocol_configuration: Optional new protocol configuration
            authorizer_configuration: Optional new authorizer configuration
            kms_key_arn: Optional new KMS key ARN
            exception_level: Optional new exception level
            client_token: Optional unique token for idempotency
            
        Returns:
            Updated gateway details
        """
        print(f"Updating AgentCore Gateway resource with ID: {gateway_id}")
        try:
            # First get current gateway details to ensure we have all required parameters
            current_gateway = agentcore_control_client.get_gateway(
                gatewayIdentifier=gateway_id
            )
            
            # Prepare update parameters with required fields from current gateway
            update_params = {
                'gatewayIdentifier': gateway_id,
                # AWS requires these parameters even for updates
                'name': name if name is not None else current_gateway['name'],
                'roleArn': role_arn if role_arn is not None else current_gateway['roleArn'],
                'protocolType': current_gateway['protocolType'],  # This cannot be changed
                'authorizerType': current_gateway['authorizerType'],  # This cannot be changed
                'authorizerConfiguration': authorizer_configuration if authorizer_configuration is not None else current_gateway.get('authorizerConfiguration', {})
            }
            
            # Add optional parameters
            if description is not None:
                update_params['description'] = description
                logger.info(f"Updating description to: {description}")
            elif current_gateway.get('description'):
                update_params['description'] = current_gateway['description']
                
            if protocol_configuration is not None:
                update_params['protocolConfiguration'] = protocol_configuration
                logger.info(f"Updating protocol configuration")
            elif current_gateway.get('protocolConfiguration'):
                update_params['protocolConfiguration'] = current_gateway['protocolConfiguration']
                
            if kms_key_arn is not None:
                update_params['kmsKeyArn'] = kms_key_arn
                logger.info(f"Updating KMS key ARN to: {kms_key_arn}")
            elif current_gateway.get('kmsKeyArn'):
                update_params['kmsKeyArn'] = current_gateway['kmsKeyArn']
                
            if exception_level is not None:
                update_params['exceptionLevel'] = exception_level
                logger.info(f"Updating exception level to: {exception_level}")
            elif current_gateway.get('exceptionLevel'):
                update_params['exceptionLevel'] = current_gateway['exceptionLevel']
                
            if client_token is not None:
                update_params['clientToken'] = client_token
                
            logger.info(f"Updating gateway with parameters: {update_params}")
            
            # Update the gateway resource
            response = agentcore_control_client.update_gateway(**update_params)
            logger.info(f"Successfully updated gateway resource with ID: {response['gatewayId']}")
            print(f"update_gateway response {response}")
            
            return {
                'gateway_id': response['gatewayId'],
                'gateway_arn': response['gatewayArn'],
                'gateway_url': response.get('gatewayUrl', ''),
                'name': response['name'],
                'description': response.get('description', ''),
                'status': response['status'],
                'role_arn': response['roleArn'],
                'protocol_type': response['protocolType'],
                'protocol_configuration': response.get('protocolConfiguration', {}),
                'authorizer_type': response['authorizerType'],
                'authorizer_configuration': response.get('authorizerConfiguration', {}),
                'created_at': response['createdAt'].isoformat() if 'createdAt' in response else None,
                'updated_at': response['updatedAt'].isoformat() if 'updatedAt' in response else None
            }
            
        except Exception as e:
            logger.error(f"Error updating Gateway resource: {str(e)}")
            raise e

    @staticmethod
    def wait_for_gateway_creation(
        gateway_id: str,
        max_attempts: int = 20,
        delay_seconds: int = 15
    ) -> Dict[str, Any]:
        """
        Wait for gateway creation to complete.
        
        Args:          
            gateway_id: Gateway ID to check
            max_attempts: Maximum number of polling attempts
            delay_seconds: Delay between polling attempts in seconds
            
        Returns:
            Gateway details when available
            
        Raises:
            TimeoutError: If the gateway creation doesn't complete within the timeout period
        """
        print(f"Called wait_for_gateway_creation for gateway {gateway_id}")
        logger.info(f"Waiting for gateway {gateway_id} to be fully created...")
        
        for attempt in range(1, max_attempts + 1):
            try:
                print(f"Attempt {attempt}")
                # Try to get the gateway details
                gateway_details = agentcore_control_client.get_gateway(
                    gatewayIdentifier=gateway_id
                )
                print(f"Got gateway details {gateway_details}")

                status = gateway_details['status']
                # Check if the gateway exists and has all expected attributes
                if status == 'READY':
                    logger.info(f"Gateway {gateway_id} is now available after {attempt} attempts")
                    print(f"wait_for_gateway_creation returning gateway_details {gateway_details}")
                    return {
                        'gateway_id': gateway_details['gatewayId'],
                        'gateway_arn': gateway_details['gatewayArn'],
                        'gateway_url': gateway_details.get('gatewayUrl', ''),
                        'name': gateway_details['name'],
                        'status': gateway_details['status'],
                        'created_at': gateway_details['createdAt'].isoformat() if 'createdAt' in gateway_details else None,
                        'updated_at': gateway_details['updatedAt'].isoformat() if 'updatedAt' in gateway_details else None
                    }
                else:
                    print(f"Gateway status: {status} (waiting {delay_seconds} seconds to check again)")
                    
            except Exception as e:
                if "Gateway not found" in str(e) or "does not exist" in str(e):
                    logger.info(f"Attempt {attempt}/{max_attempts}: Gateway {gateway_id} not yet available")
                else:
                    logger.warning(f"Unexpected error checking gateway: {str(e)}")
            
            # Wait before the next attempt
            if attempt < max_attempts:
                for t in range(1, delay_seconds + 1):
                    print('.', end='')
                time.sleep(1)
                print()  # New line after dots
        
        raise TimeoutError(f"Gateway {gateway_id} did not become available within the timeout period")

    @staticmethod
    def wait_for_gateway_deletion(
        gateway_id: str,
        max_attempts: int = 20,
        delay_seconds: int = 15
    ) -> bool:
        """
        Wait for gateway deletion to complete.
        
        Args:
            gateway_id: Gateway ID that was deleted
            max_attempts: Maximum number of polling attempts
            delay_seconds: Delay between polling attempts in seconds
            
        Returns:
            Boolean indicating if the gateway was successfully deleted
            
        Raises:
            TimeoutError: If the gateway deletion doesn't complete within the timeout period
        """
        logger.info(f"Waiting for gateway {gateway_id} to be fully deleted...")
        
        for attempt in range(1, max_attempts + 1):
            try:
                # Try to get the gateway details - this should eventually fail
                gateway_details = agentcore_control_client.get_gateway(
                    gatewayIdentifier=gateway_id
                )
                
                # If we get here, the gateway still exists
                logger.info(f"Attempt {attempt}/{max_attempts}: Gateway {gateway_id} still exists")
                
            except Exception as e:
                if "Gateway not found" in str(e) or "does not exist" in str(e):
                    logger.info(f"Gateway {gateway_id} successfully deleted after {attempt} attempts")
                    return True
                else:
                    logger.warning(f"Unexpected error checking gateway deletion: {str(e)}")
            
            # Wait before the next attempt
            if attempt < max_attempts:
                time.sleep(delay_seconds)
        
        raise TimeoutError(f"Gateway {gateway_id} was not deleted within the timeout period")


In [2]:
# first make sure the sample-agentic-platform/src is in our path

import sys
sys.path

['/Users/davetbo/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python312.zip',
 '/Users/davetbo/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12',
 '/Users/davetbo/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/lib-dynload',
 '',
 '/Users/davetbo/workplace/cline4/sample-agentic-platform/.venv/lib/python3.12/site-packages',
 '/Users/davetbo/workplace/cline4/sample-agentic-platform/src']

In [3]:
# if it's not, add it

sys.path.insert(0, '../../src')
sys.path

['../../src',
 '/Users/davetbo/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python312.zip',
 '/Users/davetbo/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12',
 '/Users/davetbo/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/lib-dynload',
 '',
 '/Users/davetbo/workplace/cline4/sample-agentic-platform/.venv/lib/python3.12/site-packages',
 '/Users/davetbo/workplace/cline4/sample-agentic-platform/src']

### Now let's create an AgentCore MCP Gateway
First we need an execution role for the gateway

In [4]:

%store -r tf_info


In [ ]:
# Create the gateway client
gateway_client = AgentCoreGatewayClient()

# Create a test MCP gateway using our AgentCoreGatewayClient
# with convenience function to create a default execution role.
discovery_url = f"https://cognito-idp.{tf_info['aws_region']['value']}.amazonaws.com/{tf_info['cognito_user_pool_id']['value']}/.well-known/openid-configuration"
cognito_client = tf_info['cognito_user_pool_client_id']['value']
gateway_response = gateway_client.create_gateway(
    name='agentpath-labs-module6-mcp-gateway',
    authorizer_configuration={
        'customJWTAuthorizer': {
            'discoveryUrl': discovery_url,
            'allowedClients': [
                cognito_client,
            ]
        }
    }
)

print(json.dumps(gateway_response, indent=2))
gateway_id = gateway_response['gateway_id']
print(f"\nCreated gateway with ID: {gateway_id}")

Creating AgentCore Gateway agentpath-labs-module6-mcp-gateway with protocol MCP
No role_arn provided, creating default IAM role...
Creating IAM role: AgentCoreGateway-agentpath-labs-module6-mcp-gateway-7e484c31
Created IAM role with ARN: arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-7e484c31
Attached policy AgentCoreGatewayPolicy-agentpath-labs-module6-mcp-gateway to role AgentCoreGateway-agentpath-labs-module6-mcp-gateway-7e484c31
Waiting for IAM role to be available...
Created default IAM role: arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-7e484c31
Creating gateway with parameters: {'name': 'agentpath-labs-module6-mcp-gateway', 'roleArn': 'arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-7e484c31', 'protocolType': 'MCP', 'authorizerType': 'CUSTOM_JWT', 'authorizerConfiguration': {'customJWTAuthorizer': {'discoveryUrl': 'https://cognito-idp.us-west-2.amazonaws.com/us-west-2_94Im

### Now let's retrieve the gateway details

In [8]:
# Get gateway details
gateway_details = gateway_client.get_gateway(gateway_id)
print("Gateway Details:")
print(json.dumps(gateway_details, indent=2))

Retrieving gateway with id agentpath-labs-module6-mcp-gateway-acqpqegfl7
Successfully retrieved gateway resource with ID: agentpath-labs-module6-mcp-gateway-acqpqegfl7
Gateway Details:
{
  "gateway_id": "agentpath-labs-module6-mcp-gateway-acqpqegfl7",
  "gateway_arn": "arn:aws:bedrock-agentcore:us-west-2:165361166149:gateway/agentpath-labs-module6-mcp-gateway-acqpqegfl7",
  "gateway_url": "https://agentpath-labs-module6-mcp-gateway-acqpqegfl7.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp",
  "name": "agentpath-labs-module6-mcp-gateway",
  "description": "",
  "status": "READY",
  "role_arn": "arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-33ea404e",
  "protocol_type": "MCP",
  "protocol_configuration": {},
  "authorizer_type": "CUSTOM_JWT",
  "authorizer_configuration": {
    "customJWTAuthorizer": {
      "discoveryUrl": "https://cognito-idp.us-west-2.amazonaws.com/us-west-2_94ImbhoL0/.well-known/openid-configuration",
      "allowedClients"

### Let's list all gateways

In [9]:
# List all gateways
gateways_list = gateway_client.list_gateways(max_results=10)
print(f"Found {len(gateways_list['gateways'])} gateways:")
for gateway in gateways_list['gateways']:
    print(f"- {gateway['name']} ({gateway['gateway_id']}) - Status: {gateway['status']}")

Listing gateways with max_results=10, next_token=None
Listing gateways with params: {'maxResults': 10}
Successfully retrieved 2 gateway resources
Got gateway: {'gatewayId': 'agentcorebuilderhubbackendstack-mcpgateway-xl4pzwbtet', 'name': 'AgentCoreBuilderHubBackendStack-McpGateway', 'status': 'READY', 'createdAt': datetime.datetime(2025, 7, 27, 17, 36, 34, 348080, tzinfo=tzutc()), 'updatedAt': datetime.datetime(2025, 7, 27, 17, 36, 34, 348090, tzinfo=tzutc()), 'authorizerType': 'CUSTOM_JWT', 'protocolType': 'MCP'}
Got gateway: {'gatewayId': 'agentpath-labs-module6-mcp-gateway-acqpqegfl7', 'name': 'agentpath-labs-module6-mcp-gateway', 'status': 'READY', 'createdAt': datetime.datetime(2025, 9, 1, 21, 52, 1, 594210, tzinfo=tzutc()), 'updatedAt': datetime.datetime(2025, 9, 1, 21, 52, 1, 594221, tzinfo=tzutc()), 'authorizerType': 'CUSTOM_JWT', 'protocolType': 'MCP'}
list_gateways returning gateways [{'gateway_id': 'agentcorebuilderhubbackendstack-mcpgateway-xl4pzwbtet', 'gateway_arn': '', '

### Now let's update the gateway configuration

In [13]:
cognito_client = tf_info['cognito_user_pool_client_id']['value']
# Update gateway description and configuration
updated_gateway = gateway_client.update_gateway(
    gateway_id=gateway_id,
    description='Updated MCP Gateway for AgentPath Labs Module 6 - Enhanced Configuration',
    authorizer_configuration={
        'customJWTAuthorizer': {
            'discoveryUrl': discovery_url,
            'allowedClients': [
                cognito_client,
            ]
        }
    }
)

print("Updated Gateway:")
print(json.dumps(updated_gateway, indent=2))

Updating AgentCore Gateway resource with ID: agentpath-labs-module6-mcp-gateway-acqpqegfl7
update_gateway response {'ResponseMetadata': {'RequestId': '58a17c7e-5499-4d25-95c0-04864ceb4ff1', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Mon, 01 Sep 2025 22:07:58 GMT', 'content-type': 'application/json', 'content-length': '1115', 'connection': 'keep-alive', 'x-amzn-requestid': '58a17c7e-5499-4d25-95c0-04864ceb4ff1', 'x-amzn-remapped-x-amzn-requestid': 'c897d702-04e6-4f6a-b59f-4ea73f05d2e5', 'x-amzn-remapped-content-length': '1115', 'x-amzn-remapped-connection': 'keep-alive', 'x-amz-apigw-id': 'QPjh0HAKvHcElRA=', 'x-amzn-trace-id': 'Root=1-68b6193e-23ece5fa4018adb468ae7060', 'x-amzn-remapped-date': 'Mon, 01 Sep 2025 22:07:58 GMT'}, 'RetryAttempts': 0}, 'gatewayArn': 'arn:aws:bedrock-agentcore:us-west-2:165361166149:gateway/agentpath-labs-module6-mcp-gateway-acqpqegfl7', 'gatewayId': 'agentpath-labs-module6-mcp-gateway-acqpqegfl7', 'gatewayUrl': 'https://agentpath-labs-module6-mcp-gatewa

### Let's verify the update by retrieving the gateway again

In [14]:
# Verify the update
updated_gateway_details = gateway_client.get_gateway(gateway_id)
print("Updated Gateway Details:")
print(json.dumps(updated_gateway_details, indent=2))

# Compare key fields
print("\n=== Comparison ===")
print(f"Description: {updated_gateway_details['description']}")
print(f"Authorizer Config: {json.dumps(updated_gateway_details['authorizer_configuration'], indent=2)}")

Retrieving gateway with id agentpath-labs-module6-mcp-gateway-acqpqegfl7
Successfully retrieved gateway resource with ID: agentpath-labs-module6-mcp-gateway-acqpqegfl7
Updated Gateway Details:
{
  "gateway_id": "agentpath-labs-module6-mcp-gateway-acqpqegfl7",
  "gateway_arn": "arn:aws:bedrock-agentcore:us-west-2:165361166149:gateway/agentpath-labs-module6-mcp-gateway-acqpqegfl7",
  "gateway_url": "https://agentpath-labs-module6-mcp-gateway-acqpqegfl7.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp",
  "name": "agentpath-labs-module6-mcp-gateway",
  "description": "Updated MCP Gateway for AgentPath Labs Module 6 - Enhanced Configuration",
  "status": "READY",
  "role_arn": "arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-33ea404e",
  "protocol_type": "MCP",
  "protocol_configuration": {},
  "authorizer_type": "CUSTOM_JWT",
  "authorizer_configuration": {
    "customJWTAuthorizer": {
      "discoveryUrl": "https://cognito-idp.us-west-2.amazonaws.c

### Now let's demonstrate error handling by trying to get a non-existent gateway

In [15]:
# Test error handling with non-existent gateway
try:
    non_existent_gateway = gateway_client.get_gateway('non-existent-gateway-id')
    print("This should not print")
except Exception as e:
    print(f"Expected error when trying to get non-existent gateway: {str(e)}")

Error retrieving AgentCore Gateway resource: An error occurred (AccessDeniedException) when calling the GetGateway operation: User: arn:aws:sts::165361166149:assumed-role/Admin/davetbo-Isengard is not authorized to perform: bedrock-agentcore:GetGateway


Retrieving gateway with id non-existent-gateway-id
Expected error when trying to get non-existent gateway: An error occurred (AccessDeniedException) when calling the GetGateway operation: User: arn:aws:sts::165361166149:assumed-role/Admin/davetbo-Isengard is not authorized to perform: bedrock-agentcore:GetGateway


### Let's create another gateway to demonstrate multiple gateways

In [16]:
# Create a second gateway with different configuration
gateway_response_2 = gateway_client.create_gateway(
    name='agentpath-labs-module6-mcp-gateway-2',
    description='Second Test MCP Gateway for AgentPath Labs Module 6',
    authorizer_configuration={
        'customJWTAuthorizer': {
            'discoveryUrl': discovery_url,
            'allowedClients': [
                cognito_client
            ]
        }
    }
)

print("Second Gateway Created:")
print(json.dumps(gateway_response_2, indent=2))
gateway_id_2 = gateway_response_2['gateway_id']
print(f"\nSecond gateway ID: {gateway_id_2}")

Creating AgentCore Gateway agentpath-labs-module6-mcp-gateway-2 with protocol MCP
No role_arn provided, creating default IAM role...
Creating IAM role: AgentCoreGateway-agentpath-labs-module6-mcp-gateway-2-7c8dcb0b
Created IAM role with ARN: arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-2-7c8dcb0b
Attached policy AgentCoreGatewayPolicy-agentpath-labs-module6-mcp-gateway-2 to role AgentCoreGateway-agentpath-labs-module6-mcp-gateway-2-7c8dcb0b
Waiting for IAM role to be available...
Created default IAM role: arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-2-7c8dcb0b
Creating gateway with parameters: {'name': 'agentpath-labs-module6-mcp-gateway-2', 'roleArn': 'arn:aws:iam::165361166149:role/AgentCoreGateway-agentpath-labs-module6-mcp-gateway-2-7c8dcb0b', 'protocolType': 'MCP', 'authorizerType': 'CUSTOM_JWT', 'description': 'Second Test MCP Gateway for AgentPath Labs Module 6', 'authorizerConfiguration': {'customJWTAut

### Now let's list all gateways again to see both

In [17]:
# List all gateways again
all_gateways = gateway_client.list_gateways()
print(f"\nTotal gateways found: {len(all_gateways['gateways'])}")
print("\nGateway Summary:")
for i, gateway in enumerate(all_gateways['gateways'], 1):
    print(f"{i}. Name: {gateway['name']}")
    print(f"   ID: {gateway['gateway_id']}")
    print(f"   Status: {gateway['status']}")
    print(f"   URL: {gateway.get('gateway_url', 'N/A')}")
    print(f"   Created: {gateway.get('created_at', 'N/A')}")
    print()

Listing gateways with max_results=None, next_token=None
Listing gateways with params: {}
Successfully retrieved 3 gateway resources
Got gateway: {'gatewayId': 'agentcorebuilderhubbackendstack-mcpgateway-xl4pzwbtet', 'name': 'AgentCoreBuilderHubBackendStack-McpGateway', 'status': 'READY', 'createdAt': datetime.datetime(2025, 7, 27, 17, 36, 34, 348080, tzinfo=tzutc()), 'updatedAt': datetime.datetime(2025, 7, 27, 17, 36, 34, 348090, tzinfo=tzutc()), 'authorizerType': 'CUSTOM_JWT', 'protocolType': 'MCP'}
Got gateway: {'gatewayId': 'agentpath-labs-module6-mcp-gateway-2-kbhdzeqhex', 'name': 'agentpath-labs-module6-mcp-gateway-2', 'status': 'READY', 'description': 'Second Test MCP Gateway for AgentPath Labs Module 6', 'createdAt': datetime.datetime(2025, 9, 1, 22, 8, 42, 38349, tzinfo=tzutc()), 'updatedAt': datetime.datetime(2025, 9, 1, 22, 8, 42, 38361, tzinfo=tzutc()), 'authorizerType': 'CUSTOM_JWT', 'protocolType': 'MCP'}
Got gateway: {'gatewayId': 'agentpath-labs-module6-mcp-gateway-acqpq

### Now let's clean up by deleting the test gateways

In [18]:
# Delete the first gateway
print(f"Deleting gateway: {gateway_id}")
delete_response_1 = gateway_client.delete_gateway(gateway_id)
print(f"Delete response: {json.dumps(delete_response_1, indent=2)}")

# Delete the second gateway
print(f"\nDeleting gateway: {gateway_id_2}")
delete_response_2 = gateway_client.delete_gateway(gateway_id_2)
print(f"Delete response: {json.dumps(delete_response_2, indent=2)}")

Deleting gateway: agentpath-labs-module6-mcp-gateway-acqpqegfl7
Deleting gateway with id agentpath-labs-module6-mcp-gateway-acqpqegfl7
Successfully deleted gateway resource with ID: agentpath-labs-module6-mcp-gateway-acqpqegfl7
Delete response: {
  "gateway_id": "agentpath-labs-module6-mcp-gateway-acqpqegfl7",
  "status": "DELETING"
}

Deleting gateway: agentpath-labs-module6-mcp-gateway-2-kbhdzeqhex
Deleting gateway with id agentpath-labs-module6-mcp-gateway-2-kbhdzeqhex
Successfully deleted gateway resource with ID: agentpath-labs-module6-mcp-gateway-2-kbhdzeqhex
Delete response: {
  "gateway_id": "agentpath-labs-module6-mcp-gateway-2-kbhdzeqhex",
  "status": "DELETING"
}


### Let's verify the gateways were deleted

In [19]:
# Wait a moment for deletion to process
import time
print("Waiting for deletion to complete...")
time.sleep(5)

# Try to get the deleted gateways (should fail)
for gw_id in [gateway_id, gateway_id_2]:
    try:
        deleted_gateway = gateway_client.get_gateway(gw_id)
        print(f"Gateway {gw_id} still exists: {deleted_gateway['status']}")
    except Exception as e:
        print(f"Gateway {gw_id} successfully deleted (expected error: {str(e)[:50]}...)")

# List gateways to confirm cleanup
final_gateways = gateway_client.list_gateways()
remaining_test_gateways = [gw for gw in final_gateways['gateways'] 
                          if 'agentpath-labs-module6' in gw['name']]
print(f"\nRemaining test gateways: {len(remaining_test_gateways)}")
if remaining_test_gateways:
    for gw in remaining_test_gateways:
        print(f"- {gw['name']} ({gw['gateway_id']}) - Status: {gw['status']}")
else:
    print("All test gateways successfully cleaned up!")

Waiting for deletion to complete...
Retrieving gateway with id agentpath-labs-module6-mcp-gateway-acqpqegfl7


Error retrieving AgentCore Gateway resource: An error occurred (ResourceNotFoundException) when calling the GetGateway operation: Failed to retrieve gateway because it doesn't exist. Retry the request with a different resource identifier.
Error retrieving AgentCore Gateway resource: An error occurred (ResourceNotFoundException) when calling the GetGateway operation: Failed to retrieve gateway because it doesn't exist. Retry the request with a different resource identifier.


Gateway agentpath-labs-module6-mcp-gateway-acqpqegfl7 successfully deleted (expected error: An error occurred (ResourceNotFoundException) when...)
Retrieving gateway with id agentpath-labs-module6-mcp-gateway-2-kbhdzeqhex
Gateway agentpath-labs-module6-mcp-gateway-2-kbhdzeqhex successfully deleted (expected error: An error occurred (ResourceNotFoundException) when...)
Listing gateways with max_results=None, next_token=None
Listing gateways with params: {}
Successfully retrieved 1 gateway resources
Got gateway: {'gatewayId': 'agentcorebuilderhubbackendstack-mcpgateway-xl4pzwbtet', 'name': 'AgentCoreBuilderHubBackendStack-McpGateway', 'status': 'READY', 'createdAt': datetime.datetime(2025, 7, 27, 17, 36, 34, 348080, tzinfo=tzutc()), 'updatedAt': datetime.datetime(2025, 7, 27, 17, 36, 34, 348090, tzinfo=tzutc()), 'authorizerType': 'CUSTOM_JWT', 'protocolType': 'MCP'}
list_gateways returning gateways [{'gateway_id': 'agentcorebuilderhubbackendstack-mcpgateway-xl4pzwbtet', 'gateway_arn': ''

## Summary

### What You've Accomplished

✅ **MCP Gateway Resource Management Operations**:
- **CREATE**: Created new MCP gateways with configurable protocols and authorization
- **READ**: Retrieved gateway details and configuration
- **UPDATE**: Modified gateway settings and configurations
- **DELETE**: Cleaned up gateway resources
- **LIST**: Enumerated all gateway resources

✅ **Gateway Management Concepts**:
- Understood MCP gateway lifecycle
- Learned about protocol configuration (MCP)
- Explored authorization mechanisms (CUSTOM_JWT)
- Practiced gateway status monitoring

✅ **Production Patterns**:
- Error handling and validation
- Resource cleanup procedures
- Proper gateway configuration
- Wait operations for async processes

### Key Learnings

1. **MCP Gateways**: Provide managed endpoints for Model Context Protocol servers with built-in security
2. **Protocol Configuration**: Flexible configuration for different MCP server endpoints and timeouts
3. **Authorization**: Support for JWT-based authentication with configurable issuers and audiences
4. **Resource Management**: Proper lifecycle management prevents resource accumulation
5. **Async Operations**: Gateway operations may be asynchronous, requiring wait patterns

### Next Steps

- **Module 03**: Explore AgentCore Runtime resource management operations
- **Module 04**: Implement end-to-end agent workflows with gateways
- **Gateway Integration**: Connect real MCP servers through gateways
- **Security Configuration**: Implement production-ready JWT authentication

---

**🎉 Congratulations! You've mastered AgentCore MCP Gateway resource management operations.**